In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import numpy as np
import random
import copy
import os
import warnings

warnings.filterwarnings("ignore")
os.makedirs('./outputs', exist_ok=True)

# ==========================================
# 1. NETWORK ARCHITECTURE (Specialists same as KFN)
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1, stride, bias=False),
                                          nn.BatchNorm2d(out_c))
    def forward(self, x): 
        return F.relu(self.bn1(self.conv1(x)) + self.shortcut(x))

class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(64, 64, 2)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.final_conv = nn.Conv2d(256, 64, 1)
    def _make_layer(self, in_c, out_c, blocks, stride=1):
        layers = [ResidualBlock(in_c, out_c, stride)]
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_c, out_c))
        return nn.Sequential(*layers)
    def forward(self, x):
        return self.final_conv(self.pool(self.layer3(self.layer2(self.layer1(self.relu(self.bn1(self.conv1(x))))))))

# ==========================================
# 2. STABILIZED MoE Global Model
# ==========================================
class MoEModel(nn.Module):
    def __init__(self, n_classes=10, n_specs=2):
        super().__init__()
        self.n_specs = n_specs
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        
        # STABLE GATING FIX: Input is concatenated features from all experts (64 * n_specs)
        self.gate_fc = nn.Linear(64 * n_specs, n_specs)  
        self.classifier = nn.Linear(64, n_classes)
    
    def forward(self, x):
        B = x.size(0)
        feats = [s(x) for s in self.specialists]  # (B,64,4,4)
        pooled = [f.mean(dim=(2,3)) for f in feats]  # list of (B,64)
        stacked = torch.stack(pooled, dim=1)  # (B, n_specs, 64)
        
        # STABLE GATING FIX: Concatenate expert features to compute logits
        gate_input = torch.cat(pooled, dim=1)  # (B, n_specs*64)
        gate_logits = self.gate_fc(gate_input)  # (B, n_specs)
        
        # Softmax over experts, properly reshaped for broadcasting
        gate_weights = F.softmax(gate_logits, dim=1).view(B, self.n_specs, 1)  # (B, n_specs, 1)
        
        # Weighted sum of expert features
        combined = (stacked * gate_weights).sum(dim=1)  # (B,64)
        out = self.classifier(combined)
        return out

# ==========================================
# 3. Utilities
# ==========================================
def set_deterministic(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

def evaluate(model, loader, device):
    model.eval(); correct=0; total=0
    with torch.no_grad():
        for x,y in loader:
            x,y = x.to(device),y.to(device)
            if getattr(device, "type", "") == 'cuda':
                with torch.amp.autocast('cuda'):
                    logits = model(x)
            else:
                logits = model(x)
            correct += (logits.argmax(1)==y).sum().item()
            total += y.size(0)
    return 100*correct/total

def get_dataset(transform_train, transform_test):
    root='/kaggle/input/cifar-10'
    train_ds = torchvision.datasets.CIFAR10(root=root, train=True, download=False, transform=transform_train)
    test_ds = torchvision.datasets.CIFAR10(root=root, train=False, download=False, transform=transform_test)
    return train_ds, test_ds

# ==========================================
# 4. MoE Training Pipeline
# ==========================================
def run_moe(seed, loaders, epochs_A=10, epochs_B=40, device=torch.device("cuda")):
    set_deterministic(seed)
    l_A, l_B, t_A, t_B, replay_loader = loaders
    
    model = MoEModel().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda') if getattr(device, "type", "") == 'cuda' else None
    
    # --- Phase 1: Task A ---
    for _ in range(epochs_A):
        model.train()
        for x,y in l_A:
            x,y = x.to(device),y.to(device)
            opt.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    loss = F.cross_entropy(model(x),y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = F.cross_entropy(model(x),y)
                loss.backward(); opt.step()
    acc_A_init = evaluate(model,t_A,device)
    
    # --- Phase 2: Task B Integration ---
    rep_iter = iter(replay_loader)
    for _ in range(epochs_B):
        model.train()
        for x_B,y_B in l_B:
            x_B,y_B = x_B.to(device),y_B.to(device)
            try:
                x_A,y_A = next(rep_iter)
            except StopIteration:
                rep_iter = iter(replay_loader)
                x_A,y_A = next(rep_iter)
            x_A,y_A = x_A.to(device), y_A.to(device)
            
            opt.zero_grad()
            if scaler:
                with torch.amp.autocast('cuda'):
                    loss_B = F.cross_entropy(model(x_B), y_B)
                    loss_A = F.cross_entropy(model(x_A), y_A)
                    # LOSS FIX: Removed aggressive 2.5 multiplier
                    loss = loss_B + 0.8 * loss_A
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss_B = F.cross_entropy(model(x_B), y_B)
                loss_A = F.cross_entropy(model(x_A), y_A)
                loss = loss_B + 0.8 * loss_A
                loss.backward(); opt.step()
    
    acc_A_final = evaluate(model,t_A,device)
    acc_B_final = evaluate(model,t_B,device)
    
    return {
        "acc_A_init": acc_A_init,
        "acc_A_final": acc_A_final,
        "acc_B_final": acc_B_final,
    }

# ==========================================
# 5. Run All Seeds & Report
# ==========================================
def run_all():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Device: {device}")
    
    stats=((0.4914,0.4822,0.4465),(0.247,0.243,0.261))
    t_train=transforms.Compose([
        transforms.RandomCrop(32,padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2),
        transforms.ToTensor(),
        transforms.Normalize(*stats)
    ])
    t_test=transforms.Compose([transforms.ToTensor(),transforms.Normalize(*stats)])
    
    train_ds,test_ds = get_dataset(t_train,t_test)
    
    EPOCHS_A=10; EPOCHS_B=40; BS=256; NW=2; REPLAY_SIZE=12000
    
    loaders=(
        DataLoader(Subset(train_ds,[i for i,l in enumerate(train_ds.targets) if l<5]),BS,shuffle=True,num_workers=NW,pin_memory=True,persistent_workers=True),
        DataLoader(Subset(train_ds,[i for i,l in enumerate(train_ds.targets) if l>=5]),BS,shuffle=True,num_workers=NW,pin_memory=True,persistent_workers=True),
        DataLoader(Subset(test_ds,[i for i,l in enumerate(test_ds.targets) if l<5]),BS,num_workers=NW,pin_memory=True),
        DataLoader(Subset(test_ds,[i for i,l in enumerate(test_ds.targets) if l>=5]),BS,num_workers=NW,pin_memory=True),
        DataLoader(Subset(train_ds,[i for i,l in enumerate(train_ds.targets) if l<5][:REPLAY_SIZE]),BS,shuffle=True,num_workers=NW,pin_memory=True,persistent_workers=True)
    )
    
    seeds=[0,1,2,3,4]
    results=[]
    
    print("\n[1] Running Multi-Seed MoE Experiments...")
    for s in seeds:
        print(f" -> Training MoE Baseline Seed {s}...")
        r = run_moe(s, loaders, EPOCHS_A, EPOCHS_B, device)
        results.append(r)
        print(f"    ↳ Seed {s} Results: Task A Init = {r['acc_A_init']:.2f}% | Task A Final = {r['acc_A_final']:.2f}% | Task B Final = {r['acc_B_final']:.2f}%")
    
    # ==========================================
    # FINAL PAPER-READY REPORTING
    # ==========================================
    print("\n" + "="*80)
    print("SECTION A: MoE Baseline (Mean ± Std over 5 Seeds)")
    print("="*80)
    
    for k, label in [("acc_A_init", "Task A Init"), ("acc_A_final", "Task A Final"), ("acc_B_final", "Task B Final")]:
        vals = [r[k] for r in results]
        print(f"{label:15}: {np.mean(vals):.2f}% ± {np.std(vals):.2f}")

    ret_vals = [(r['acc_A_final']/r['acc_A_init'])*100 for r in results]
    forg_vals = [r['acc_A_init'] - r['acc_A_final'] for r in results]
    bwt_vals = [r['acc_A_final'] - r['acc_A_init'] for r in results]

    print(f"Retention:      {np.mean(ret_vals):.2f}% ± {np.std(ret_vals):.2f}")
    print(f"Forgetting:     {np.mean(forg_vals):.2f} ± {np.std(forg_vals):.2f}")
    print(f"Backward Trans: {np.mean(bwt_vals):.2f} ± {np.std(bwt_vals):.2f}")
    print("="*80)

if __name__=="__main__":
    run_all()

🚀 Device: cuda

[1] Running Multi-Seed MoE Experiments...
 -> Training MoE Baseline Seed 0...
    ↳ Seed 0 Results: Task A Init = 82.88% | Task A Final = 81.76% | Task B Final = 52.72%
 -> Training MoE Baseline Seed 1...
    ↳ Seed 1 Results: Task A Init = 81.08% | Task A Final = 86.70% | Task B Final = 36.74%
 -> Training MoE Baseline Seed 2...
    ↳ Seed 2 Results: Task A Init = 79.00% | Task A Final = 85.62% | Task B Final = 53.18%
 -> Training MoE Baseline Seed 3...
    ↳ Seed 3 Results: Task A Init = 81.60% | Task A Final = 79.96% | Task B Final = 51.92%
 -> Training MoE Baseline Seed 4...
    ↳ Seed 4 Results: Task A Init = 81.76% | Task A Final = 73.38% | Task B Final = 59.70%

SECTION A: MoE Baseline (Mean ± Std over 5 Seeds)
Task A Init    : 81.26% ± 1.28
Task A Final   : 81.48% ± 4.74
Task B Final   : 50.85% ± 7.58
Retention:      100.34% ± 6.76
Forgetting:     -0.22 ± 5.47
Backward Trans: 0.22 ± 5.47
